In [0]:
data = [
    (1, "A", 5000),
    (2, "B", 6000),
    (3, "C", 7000)
]

columns = ["id", "name", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .save("/Volumes/workspace/default/day1")

## READ DELTA TABLE

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/day1")
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


### INSERT

In [0]:
new_data = [(4, "D", 8000)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/workspace/default/day1")

### UPDATE

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/workspace/default/day1")

delta_table.update(
    condition = "id = 2",
    set = {"salary": "6500"}
)

DataFrame[num_affected_rows: bigint]

### DELETE

In [0]:
delta_table.delete("id = 1")

DataFrame[num_affected_rows: bigint]

### MERGE INTO

#### existing data

In [0]:
target_df = spark.read.format("delta").load("/Volumes/workspace/default/day1")
target_df.display()

id,name,salary
2,B,6500
3,C,7000
4,D,8000


#### new data

In [0]:
updates = [
    (2, "B", 7000),  # update
    (5, "E", 9000)   # insert
]
columns=["id", "name", "salary"]
updates_df = spark.createDataFrame(updates, columns)
updates_df.display()

id,name,salary
2,B,7000
5,E,9000


#### merge

In [0]:
delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "salary": "source.salary"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "name": "source.name",
    "salary": "source.salary"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

#### If id exists → UPDATE
#### If id not exists → INSERT
#### This is UPSERT

#### Problem:
#### New column arrives
#### example :

In [0]:
new_data = [(6, "F", 10000, "India")]
new_columns = ["id", "name", "salary", "country"]

df_new = spark.createDataFrame(new_data, new_columns)
df_new.display()

id,name,salary,country
6,F,10000,India


#### enable schema evolution

In [0]:
df_new.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/day1")

In [0]:
df_read=spark.read.format('delta').load("/Volumes/workspace/default/day1")
df_read.display()

id,name,salary
3,C,7000
4,D,8000
2,B,7000
5,E,9000


### Time Travel

#### View History

In [0]:
delta_table.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-04-16T06:00:58.000Z,73256479438112,bhavyasrilakshmipalla2007@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3884942927263047),6159e234-662c-4b0d-a212-45a2793c279a,0416-054311-lc4229x5-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3053, p25FileSize -> 1083, numDeletionVectorsRemoved -> 1, minFileSize -> 1083, numAddedFiles -> 1, maxFileSize -> 1083, p75FileSize -> 1083, p50FileSize -> 1083, numAddedBytes -> 1083)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-16T06:00:55.000Z,73256479438112,bhavyasrilakshmipalla2007@gmail.com,MERGE,"Map(predicate -> [""(id#11827L = id#11830L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3884942927263047),6159e234-662c-4b0d-a212-45a2793c279a,0416-054311-lc4229x5-v2n,6,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2008, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5949, materializeSourceTimeMs -> 672, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2038, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3082)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-04-16T05:57:27.000Z,73256479438112,bhavyasrilakshmipalla2007@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3884942927263047),8d22e24a-1516-4d70-be02-541f1b04bdd8,0416-054311-lc4229x5-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1061, p25FileSize -> 1045, numDeletionVectorsRemoved -> 1, minFileSize -> 1045, numAddedFiles -> 1, maxFileSize -> 1045, p75FileSize -> 1045, p50FileSize -> 1045, numAddedBytes -> 1045)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-04-16T05:57:25.000Z,73256479438112,bhavyasrilakshmipalla2007@gmail.com,DELETE,"Map(predicate -> [""(id#11510L = 1)""])",null,List(3884942927263047),8d22e24a-1516-4d70-be02-541f1b04bdd8,0416-054311-lc4229x5-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1865, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1231, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 633)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-04-16T05:57:05.000Z,73256479438112,bhavyasrilakshmipalla2007@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3884942927263047),07c138c8-05d5-4320-af03-c2f8078b7f51,0416-054311-lc4229x5-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3051, p25FileSize -> 1061, numDeletionVectorsRemoved -> 1, minFileSize -> 1061, numAddedFiles -> 1, maxFileSize -> 1061, p75FileSize -> 1061, p50FileSize -> 1061, numAddedBytes -> 1061)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-04-16T05:57:02.000Z,73256479438112,bhavyasrilakshmipalla2007@gmail.com,UPDATE,"Map(predicate -> [""(id#11166L = 2)""])",null,List(3884942927263047),07c138c8-05d5-4320-af03-c2f8078b7f51,0416-054311-lc4229x5-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -

#### Read Old Version

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/day1")
df_old.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


#### restore Table

In [0]:
delta_table.restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]